# BAB V. Eksplorasi Karakteristik Data (Exploring Data Characteristics)

Statistik hanya dihitung untuk kolom yang telah diverifikasi bertipe numerik dan memiliki interpretasi yang jelas. Statistik deskriptif tidak menggantikan pemahaman satuan dan konteks variabel.

In [ ]:
from pathlib import Path
import pandas as pd
from eda_knkt.config import PROCESSED_DATA_DIR
from eda_knkt.ingestion import load_csv
from eda_knkt.cleaning import replace_placeholders
from eda_knkt.transformation import convert_numeric_columns, add_distance_to_stop_feature

DATA_FILE = PROCESSED_DATA_DIR / 'data_logger_knkt_tabel_6_10.csv'
df_raw = load_csv(DATA_FILE)
df_missing = replace_placeholders(df_raw, ['---'])
NUMERIC_OPERATIONAL_COLUMNS = [
    'km', 'm', 'Speed (km/h)', 'Acceleration (m/sec²)',
    'Load (amps)', 'EAB BP (psi)', 'EAB BC (psi)'
]
df_transformed = convert_numeric_columns(
    df_missing, NUMERIC_OPERATIONAL_COLUMNS, errors='coerce'
)
df_transformed = add_distance_to_stop_feature(df_transformed)
df_transformed['Timestamp'] = pd.to_datetime(
    df_transformed['Date'].astype(str) + ' ' + df_transformed['Time'].astype(str),
    format='%d/%m/%Y %H.%M.%S', errors='raise'
)
df_transformed = df_transformed.sort_values('Timestamp').reset_index(drop=True)
df_transformed['Elapsed (s)'] = (
    df_transformed['Timestamp'] - df_transformed['Timestamp'].iloc[0]
).dt.total_seconds()
print(f'Loaded {len(df_raw)} records × {len(df_raw.columns)} columns from {DATA_FILE.name}')

In [ ]:
from eda_knkt.statistics import numeric_summary, categorical_summary, distribution_shape

if "df_missing" not in globals():
    print("Jalankan notebook 03 dan 04 terlebih dahulu.")
else:
    numeric_columns = [c for c in df_transformed.select_dtypes(include="number").columns if c not in {"km", "m"}]
    categorical_columns = ["Throttle", "PCS Open", "IBS", "DB Excit", "Dir Call", "DB Set", "DB Star"]
    print("Numeric columns:", numeric_columns)
    print("Categorical columns:", categorical_columns)
    if numeric_columns:
        display(numeric_summary(df_transformed, numeric_columns))
        display(distribution_shape(df_transformed, numeric_columns))
    if categorical_columns:
        display(categorical_summary(df_transformed.replace("", pd.NA), categorical_columns))

## Interpretasi
- Mean dan median menggambarkan pemusatan; bandingkan keduanya dengan bentuk distribusi.
- Standar deviasi, varians, rentang, dan kuartil menggambarkan penyebaran.
- Skewness menunjukkan kemencengan distribusi.
- Kurtosis Pandas merupakan *excess kurtosis*; nilai acuan distribusi normal adalah sekitar 0, bukan 3.
- Statistik harus ditafsirkan dalam satuan variabel yang benar.

Jangan menyatakan data normal hanya berdasarkan skewness/kurtosis. Gunakan visualisasi dan pertimbangkan ukuran sampel serta konteks pengukuran.